In [1]:
import os
import time
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
from statsmodels.tsa.api import ARIMA, ExponentialSmoothing
import torch
from gym import spaces

import warnings

from blockhouse_ml.utils.macro_model import MetaLearner
from blockhouse_ml.utils.data_handler import DataProcessor, InferenceDataHandler
from blockhouse_ml.utils.macro_model_utils import MacroTraderModel
from blockhouse_ml.utils import fetch_merge_data
from blockhouse_ml.utils.env import TradingEnvironment, CustomTradingEnvironment
from blockhouse_ml.utils.benchmark_utils import simulate_twap_strategy, simulate_vwap_strategy, simulate_our_strategy

c:\Users\yashv\miniconda3\envs\MLProj\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-08-21 12:32:26,798	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2024-08-21 12:32:27,065	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
warnings.filterwarnings("ignore")

In [3]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
'''
Because of this error:
 [error] Disposing session as kernel process died ExitCode: 3, Reason: OMP: Error #15: Initializing libiomp5md.dll, but found libomp140.x86_64.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.
'''

'\nBecause of this error:\n [error] Disposing session as kernel process died ExitCode: 3, Reason: OMP: Error #15: Initializing libiomp5md.dll, but found libomp140.x86_64.dll already initialized.\nOMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.\n'

In [4]:
# Initialize the model directory, where the models will be saved
MODEL_DIR = 'Models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Initialize the data directory, where the data will be stored
data_dir = 'Data'
os.makedirs(data_dir, exist_ok=True)

# Initialize the MetaLearner
meta = MetaLearner()

# Initialize the MacroTraderModel
macro_trader = MacroTraderModel(MODEL_DIR)

# Initialize the data processor
data_processor = DataProcessor()

# Inference data handler
inference_data_handler = InferenceDataHandler()

# Define the forecast steps
forecast_steps = {
    'open': (360, '1T'),
    'high': (360, '1T'),
    'low': (360, '1T'),
    'close': (360, '1T'),
    'volatility': (360, '1T'),
    'volume': (360, '1T'),
    'transaction_cost': (360, '1T')
}



# Define the start and end dates
start_time = '2024-07-01'
end_time = '2024-08-16'

# List of Companies based on Market Capitalization
large_cap_companies = ['AAPL', 'CSCO', 'MCD', 'IBM', 'AMZN', 'TSLA', 'PFE', 'MS','MSFT','NVDA']
mid_cap_companies = [] #['AEG', 'NICE', 'NLY', 'ONTO', 'PSN', 'SAIA', 'OWL','PNW','TWLO','HAS']
small_cap_companies = [] #['NVAX','AMC','WOLF','IREN','SEDG', 'UPWK','SERV','FSLY','BMBL','ARRY']

In [5]:
def get_data(data_dir, ticker, start_time, end_time):
     ## Fetch and process the data and save it to the data directory, if it doesn't exist
    filename = f'{data_dir}/merged_data_{ticker}_{start_time}_{end_time}.csv'
    if os.path.exists(filename):
        data = pd.read_csv(filename)
    else:
        data = fetch_merge_data.fetch_and_merge_data(ticker,start_date=start_time,end_date=end_time,save_dir =data_dir)

    processed_filename = f'{data_dir}/processed_data_{ticker}_{start_time}_{end_time}.csv'

    if os.path.exists(processed_filename):
        processed_data = pd.read_csv(processed_filename)
    else:
        processed_data = data_processor.process_data(data, forecast_steps,n_jobs=4)
        processed_data.to_csv(processed_filename)

    return processed_data

In [6]:
from collections import defaultdict
results_twap = defaultdict() # Dictionary to store results from TWAP, VWAP and OUR model
results_our = defaultdict()
results_vwap = defaultdict()

IC_twap = defaultdict() # Dictionary to store IC from TWAP model for each ticker

ticker = "TSLA"
processed_data = get_data(data_dir, ticker, start_time, end_time)
# Only need test data for testing
test_data = processed_data[int(len(processed_data) * 0.8):]

print("Testing Model for ", ticker)
market_cap_int = fetch_merge_data.get_market_cap(ticker)
market_cap = meta.classify_market_cap(market_cap_int)
timeframe=500
inventory=10000
# for transaction_size in [9, 99, 499, 1999, 10000]:
scenario = meta.classify_scenario(inventory)
#     print("Training Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
#     training_params['tb_log_name'] =f'{log_dir}/{ticker}_{scenerio}_{market_cap}'
print("------------------------")
print("Testing on ", ticker)
_, tenv = macro_trader.test(test_data, market_cap=market_cap, scenario=scenario)
# print("------------------------")
print("Testing on TWAP | ",end="")
slippage , market, liquidity, tc, twap_rew = simulate_twap_strategy(data=test_data,initial_inventory=inventory,preferred_timeframe=timeframe)
results_twap[ticker] = (slippage,market,liquidity,tc, twap_rew)
IC_twap[ticker] = twap_rew
print("slippage on TWAP: ", sum(slippage)/len(slippage))
print("Reward on TWAP: ", twap_rew)
# print("------------------------")
print("Testing on VWAP | ",end="")
slippage , market, liquidity, tc, vwap_rew = simulate_vwap_strategy(data=test_data,initial_inventory=inventory,preferred_timeframe=timeframe)
results_vwap[ticker] = (slippage,market,liquidity,tc, vwap_rew)
print("slippage on VWAP: ", sum(slippage)/len(slippage))
print("Reward on VWAP: ", vwap_rew)    
# print("------------------------")
print("Testing on OUR | ",end="")
trades = pd.DataFrame(tenv.trades)
slippage , market, liquidity, tc, our_rew = simulate_our_strategy(trades, data=test_data)
results_our[ticker] = (slippage,market,liquidity,tc, our_rew)
print("slippage on OUR: ", sum(slippage)/len(slippage))
print("Reward on OUR: ", our_rew)
# print(" total trades : ", len(trades))
print("------------------------")


Testing Model for  TSLA
------------------------
Testing on  TSLA
large cap, large scenario model selected
Testing on TWAP | slippage on TWAP:  -8.451807599999992
Reward on TWAP:  (-8.301157664797897, 38.419886141788545)
Testing on VWAP | slippage on VWAP:  -8.451807599999992
Reward on VWAP:  (-8.301173399530567, 38.41988674968025)
Testing on OUR | slippage on OUR:  -191.774
Reward on OUR:  (-191.7621287565035, 0.2863691980071277)
------------------------


In [13]:
trades

,step,action,price,shares,reward,inventory,time left
0,44,"[0.3206227, 43.093437]",192.12,3207,-1.175563,6793,302
1,74,"[0.33, 30.0]",192.55,2242,-0.305985,4551,242
2,112,"[0.32855126, 37.53511]",192.22,1496,1.036576,3055,166
3,162,"[0.33, 49.012943]",189.86,1009,-0.327122,2046,66
4,212,"[0.06222964, 50.0]",191.35,128,0.267442,1918,-34


In [14]:
current_step = 0
print(test_data.iloc[current_step][['close', 'expected_price']])
for step in trades['step']:
    current_step += step
    print(test_data.iloc[current_step][['close', 'expected_price']])


close             192.12
expected_price       0.0
Name: 25053, dtype: object
close             192.55
expected_price     192.5
Name: 25097, dtype: object
close             190.58
expected_price    190.56
Name: 25171, dtype: object
close              190.6
expected_price    191.05
Name: 25283, dtype: object
close             199.16
expected_price    199.38
Name: 25445, dtype: object
close             199.7775
expected_price      199.44
Name: 25657, dtype: object


In [7]:
# Getting the summary table
delta_IC_metrics = defaultdict(list)
IC_std_metrics = defaultdict(list)

slipage_metrics = defaultdict(list)

market_metrics = defaultdict(list)

for result_dict in [results_twap, results_vwap, results_our]:
    for ticker, values in result_dict.items():
        
        IC_twap_mean, IC_twap_std = IC_twap[ticker]
        IC_mean, IC_std = values[-1]
        delta_IC = IC_mean - IC_twap_mean
        slippage, market, liquidity, tc, _ = values
        slipage_metrics[ticker].append(sum(slippage)/len(slippage))
        market_metrics[ticker].append(sum(market)/len(market))
        # print(f"ticker : {ticker} | slippage : {sum(slippage)} | market : {sum(market)} | liquidity : {sum(liquidity)} | tc : {sum(tc)} | IC : {IC_mean}")
        # print(f"Ticker: {ticker} | delta_IC: {delta_IC} | IC_twap_mean: {IC_twap_mean} | IC_mean: {IC_mean} | IC_std: {IC_std}")
        # delta_IC_std = np.sqrt(IC_std**2 + IC_twap_std**2)
        # for slippage, market, liquidity, tc, IC in values:
        #     IC_mean, IC_std = IC
        #     slippages.append(slippage)
        delta_IC_metrics[ticker].append(delta_IC)
        IC_std_metrics[ticker].append(IC_std)

# Creating a DataFrame for the summary table
delta_IC_metrics_df = pd.DataFrame(delta_IC_metrics, index=["TWAP", "VWAP", "OUR"])
IC_std_metrics_df = pd.DataFrame(IC_std_metrics, index=["TWAP", "VWAP", "OUR"])
slippage_metrics_df = pd.DataFrame(slipage_metrics, index=["TWAP", "VWAP", "OUR"])
market_metrics_df = pd.DataFrame(market_metrics, index=["TWAP", "VWAP", "OUR"])

delta_IC_metrics_df.head()

,TSLA
TWAP,0.000000
VWAP,-0.000016
OUR,-183.460971


In [9]:
ticker = "TSLA"
processed_data = get_data(data_dir, ticker, start_time, end_time)

test_data = processed_data[int(len(processed_data) * 0.8):]
test_data.head()

,Unnamed: 0,timestamp,datetime,open,high,low,close,volume,ask_price,ask_size,...,5_min_volatility,5_min_volume,5_min_TC,forecast_6Hr_open,forecast_6Hr_high,forecast_6Hr_low,forecast_6Hr_close,forecast_6Hr_volatility,forecast_6Hr_volume,forecast_6Hr_transaction_cost
25053,25094,1723104360000,2024-08-08 08:06:00,191.98,192.25,191.98,192.12,4104.0,0.0,0,...,0.000426,41360.0,0.015792,231.332889,197.485734,248.167760,243.758725,0.010059,17.348129,0.063823
25054,25095,1723104420000,2024-08-08 08:07:00,192.09,192.21,192.07,192.15,4713.0,0.0,0,...,0.000342,34634.0,0.018089,196.358096,196.901858,245.029117,238.258572,0.010139,16.777649,0.041745
25055,25096,1723104480000,2024-08-08 08:08:00,192.10,192.13,191.99,192.07,4529.0,0.0,0,...,0.000142,25078.0,0.020909,196.202061,196.446144,234.597544,229.903186,0.010036,16.210289,0.018821
25056,25097,1723104540000,2024-08-08 08:09:00,191.99,191.99,191.65,191.65,8378.0,0.0,0,...,0.000134,27502.0,0.019398,195.624030,195.513683,194.167774,193.987492,0.010377,16.552992,0.022214
25057,25098,1723104600000,2024-08-08 08:10:00,191.60,191.62,191.20,191.41,6710.0,0.0,0,...,0.000222,28434.0,0.016653,193.556072,192.901124,191.379997,192.309997,0.010654,16.511251,0.022669


In [10]:
test_data.describe()

,Unnamed: 0,timestamp,open,high,low,close,volume,ask_price,ask_size,bid_price,...,5_min_volatility,5_min_volume,5_min_TC,forecast_6Hr_open,forecast_6Hr_high,forecast_6Hr_low,forecast_6Hr_close,forecast_6Hr_volatility,forecast_6Hr_volume,forecast_6Hr_transaction_cost
count,6264.000000,6.264000e+03,6264.000000,6264.000000,6264.000000,6264.000000,6.264000e+03,6264.000000,6264.000000,6264.000000,...,6264.000000,6.264000e+03,6264.000000,6264.000000,6264.000000,6264.000000,6264.000000,6264.000000,6264.000000,6264.000000
mean,28225.500000,1.723512e+12,203.731108,203.841017,203.623215,203.733774,7.833419e+04,197.423266,49.520913,197.331074,...,0.000248,3.916828e+05,0.156552,205.398291,205.043358,204.628823,205.493670,0.003165,9.840596,0.204955
std,1808.405375,2.393665e+08,7.154518,7.149722,7.158162,7.154211,1.326517e+05,35.909024,328.332582,35.895713,...,0.000386,5.736822e+05,0.229257,17.912455,17.735318,19.375430,17.207718,0.002472,10.354415,8.454395
min,25094.000000,1.723104e+12,189.790000,189.800000,189.630000,189.790000,1.250000e+02,0.000000,0.000000,0.000000,...,0.000003,1.329000e+03,0.000531,-106.142642,81.299117,-37.802077,53.180198,-0.001776,-86.592716,-90.078900
25%,26659.750000,1.723230e+12,198.400000,198.550000,198.250000,198.400000,1.629500e+03,197.940000,0.000000,197.895000,...,0.000073,9.975250e+03,0.004013,198.101165,197.811080,197.945974,198.121361,0.001775,5.792294,-0.262216
50%,28225.500000,1.723564e+12,200.960000,201.000000,200.916150,200.944100,9.449000e+03,200.870000,0.000000,200.710000,...,0.000147,5.164400e+04,0.020664,203.743236,203.376732,203.526490,203.776369,0.002607,9.834360,0.003422
75%,29791.250000,1.723722e+12,207.952500,208.155000,207.830100,208.000000,1.151565e+05,207.690000,14.000000,207.570000,...,0.000278,5.979808e+05,0.238993,210.964176,210.718488,210.874552,211.040758,0.003749,12.957128,0.115018
max,31357.000000,1.723849e+12,219.659300,219.800000,219.410000,219.659900,3.601058e+06,217.890000,13302.000000,217.790000,...,0.005551,4.924110e+06,1.965844,557.865361,522.479242,525.399996,483.995796,0.053238,210.194085,171.612456
